<a href="https://colab.research.google.com/github/eluan216/ecg-arrhythmia-detection/blob/main/ecg_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ECG Arrhythmia Detection — Google Colab Runner

Run the [ecg-arrhythmia-detection](https://github.com/eluan216/ecg-arrhythmia-detection) pipeline on Colab.

**Recommended:** clone from GitHub (Cell 1). Zip upload still works if you prefer.

> Research / educational only — **NOT** a medical diagnostic device.


## 1 — Clone repository


In [ ]:
import os
os.chdir("/content")
!git clone --depth 1 https://github.com/eluan216/ecg-arrhythmia-detection.git
%cd /content/ecg-arrhythmia-detection
!ls -la


### Alternative: unzip a local zip (skip if you cloned)

Upload `ecg-arrhythmia-detection-main.zip` via the Files sidebar, then run:


In [ ]:
# import zipfile, os
# os.chdir("/content")
# !unzip -o ecg-arrhythmia-detection-main.zip
# os.chdir("/content/ecg-arrhythmia-detection-main")
# !ls


## 2 — Install dependencies

Colab already has NumPy / scikit-learn / PyTorch. Install project extras:


In [ ]:
!pip install -q "wfdb>=4.1.0" "mlflow>=2.0" "fastapi>=0.104" "uvicorn[standard]>=0.24" "pytest>=7.0" joblib
# optional boosting models for the benchmark table:
!pip install -q xgboost lightgbm
!python -c "import wfdb, sklearn, torch; print('wfdb', wfdb.__version__, '| torch', torch.__version__)"


## 3 — Offline multi-model benchmark (no MIT-BIH download)

Synthetic AAMI-style beats → morphology features → LogReg / RF / (XGB/LGBM) → stratified CV.  
Writes `models/benchmark_results.json`. ~1 minute.


In [ ]:
!python -m src.pipeline.benchmark --samples 150


## 4 — Random Forest baseline

Uses **real MIT-BIH** if `data/raw` has records; otherwise **automatically falls back to synthetic features** so this cell never fails with "Failed to load any records".


In [ ]:
!python -m src.train.train_baseline
from pathlib import Path
import json
p = Path("models/baseline_metrics.json")
print(json.dumps(json.loads(p.read_text()), indent=2) if p.exists() else "metrics file missing")


## 5 — Full MIT-BIH download + baseline on real data (optional, slower)

Downloads PhysioNet MIT-BIH into `data/raw/` then re-trains the baseline on real inter-patient splits.


In [ ]:
!python -m src.data.download
from src.data.download import get_record_info
print(get_record_info("data/raw"))
!python -m src.train.train_baseline


## 6 — 1D-CNN + comparison (optional GPU)

**Runtime → Change runtime type → T4 GPU** for faster training.


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!python -m src.train.train_cnn
!python -m src.evaluate


## 7 — Unit tests (optional)


In [ ]:
!python -m pytest tests/ -q -o addopts= 2>/dev/null || python -m pytest tests/ -q --no-cov
